In [1]:
import pandas as pd
import json

In [2]:
mimic = pd.read_csv("./data/clinical_cases.csv")

In [84]:
count = 0
cases = []
for row in mimic.iterrows():
    row = row[1]
    print(row)
    clinical_case = {"diagnosis": row["admission_diagnosis"], 
                     "age": row["age"], 
                     "gender": row["gender"],
                     "caseId": row["case_id"],
                     "subjectId": row["subject_id"],
                     "hadmId": row["hadm_id"]
                     }
    summary = row["discharge_summary"]
    sections = summary.split("\n\n")
    for section in sections:
        section = section.strip()
        if len(section) > 13 and section[:13].lower() == "physical exam":
            clinical_case["physicalExamination"] = section
        if len(section) > 20 and section[:20].lower() == "past medical history":
            clinical_case["pastMedicalHistory"] = section
        if len(section) > 19 and section[:19].lower() == "medications at home":
            clinical_case["medicationsAtHome"] = section
        if len(section) > 9 and section[:9].lower == "allergies":
            clinical_case["allergies"] = section
    if "physicalExamination" in clinical_case and "pastMedicalHistory" in clinical_case:
        cases.append(clinical_case)
    if len(cases) >= 10:
        break

case_id                                                       CASE_00001
subject_id                                                          5857
hadm_id                                                         149568.0
age                                                                   56
gender                                                                 F
admission_diagnosis                                    COPD EXACERBATION
discharge_summary      Admission Date:  [**2184-1-21**]     Discharge...
Name: 0, dtype: object
case_id                                                       CASE_00002
subject_id                                                          5525
hadm_id                                                         132331.0
age                                                                   66
gender                                                                 M
admission_diagnosis                                  GANGRENE RIGHT FOOT
discharge_summary      Admis

In [51]:
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Dict, List

In [ ]:
client = OpenAI(api_key=ADD API KEY HERE)

In [89]:
class NamedValue(BaseModel):
    key: str
    value: str
    
class PhysicalExamination(BaseModel):
    vitals: List[NamedValue] = Field(default_factory=list)
    findings: List[NamedValue] = Field(default_factory=list)
     
class Background(BaseModel):
    physicalExamination: PhysicalExamination = Field(default_factory=PhysicalExamination)
    pastMedicalHistory: List[str] = Field(default_factory=list)
    incorrectChoices: List[str] = Field(default_factory=list)
    

In [93]:
backgrounds = []
jsonOutput = []
for case in cases:
    response = client.responses.parse(
    model="gpt-5.4-2026-03-05",
    input = [
        {"role": "system", "content": """We are parsing the the background discharge summary
         so that it can be used by somebody to diagnose the patient. There can be different
         types of vitals and other information. You can add whichever information you find in the
         context that is relevant, to your response. Also you have to add 3 incorrect choices in 
         the incorrectChoicesField"""},
        {"role": "user", "content": f"{case["pastMedicalHistory"]}\n\n{case["physicalExamination"]}"}
    ],
    text_format=Background
    )
    background = response.output_parsed
    if background is None:
        continue
    backgroundJson = {"diagnosis": case["diagnosis"], 
                     "age": case["age"], 
                     "gender": case["gender"],
                     "caseId": case["caseId"],
                     "subjectId": case["subjectId"],
                     "hadmId": case["hadmId"]}
    print(background)
    backgroundJson["incorrectChoices"] = background.incorrectChoices
    backgroundJson["pastMedicalHistory"] = background.pastMedicalHistory
    backgroundJson["physicalExamination"] = {"vitals": [], "findings": []}
    for vital in background.physicalExamination.vitals:
        backgroundJson["physicalExamination"]["vitals"].append({vital.key: vital.value})
    for finding in background.physicalExamination.findings:
        backgroundJson["physicalExamination"]["findings"].append({finding.key: finding.value})
    jsonOutput.append(backgroundJson)
    

physicalExamination=PhysicalExamination(vitals=[NamedValue(key='Temperature', value='98.2'), NamedValue(key='Blood pressure', value='139/72'), NamedValue(key='Pulse', value='84'), NamedValue(key='Weight', value='79 kg'), NamedValue(key='Oxygen support', value='Assist control ventilation on ICU arrival'), NamedValue(key='Tidal volume', value='600'), NamedValue(key='FiO2', value='40%'), NamedValue(key='ABG', value='pH 7.48 / pCO2 46 / pO2 110')], findings=[NamedValue(key='General', value='No acute distress, awake, responded appropriately with nods'), NamedValue(key='HEENT', value='Pale conjunctivae; extraocular muscles intact; pupils equal, round, and reactive to light; no lymphadenopathy; obese neck; oropharynx difficult to visualize with ET tube but no evidence of lesions'), NamedValue(key='Cardiovascular', value='S1 and S2 present, regular rate and rhythm'), NamedValue(key='Pulmonary', value='Rare expiratory wheeze'), NamedValue(key='Abdomen', value='Soft, nontender, nondistended, pos

In [94]:
with open("clinicalCases.json", "w") as f:
    json.dump(jsonOutput, f)

In [72]:
print(response.output_parsed.physicalExamination.findings)

[NamedValue(key='General', value='No acute distress, awake, responded appropriately with nods'), NamedValue(key='HEENT', value='Pale conjunctivae; extraocular muscles intact; pupils equal, round, and reactive to light; no lymphadenopathy; obese neck; oropharynx difficult to visualize due to ET tube, no evidence of lesions'), NamedValue(key='Cardiovascular', value='S1 and S2 present, regular rate and rhythm'), NamedValue(key='Pulmonary', value='Rare expiratory wheeze'), NamedValue(key='Abdomen', value='Soft, nontender, nondistended, positive bowel sounds'), NamedValue(key='Extremities', value='Trace pitting edema to calves'), NamedValue(key='Neurologic', value='Grossly intact')]
